In [29]:
import pandas as pd
df = pd.read_csv(r"D:\GIT\data-analytics-portfolio\01-world-bank-analysis\data\raw_data\africa_data.csv", low_memory=False)


print(df.shape)

(68908, 80)


In [30]:
# Selecting relevant columns

cols = [
    'Country Name',
    'Country Code',
    'Indicator Name',
    'Indicator Code'
] + [str(year) for year in range(2000, 2025)]

df = df[cols]

In [31]:
df.head()

,Country Name,Country Code,Indicator Name,Indicator Code,2000,2001,2002,2003,2004,2005,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,Algeria,DZA,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.ZS,96.5,97.0,97.45,97.8,98.1,98.4,...,99.6,99.6,99.6,99.7,99.7,99.7,99.7,99.7,99.7,NaN
1,Algeria,DZA,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.RU.ZS,92.6,93.4,94.10,94.8,95.4,95.9,...,98.5,98.6,98.8,98.8,98.9,98.9,99.0,99.1,99.1,NaN
2,Algeria,DZA,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.UR.ZS,99.6,99.7,99.80,99.8,99.9,99.9,...,99.9,99.9,100.0,100.0,100.0,99.9,99.9,99.9,99.9,NaN
3,Algeria,DZA,Access to electricity (% of population),EG.ELC.ACCS.ZS,98.6,98.6,98.60,98.6,98.6,98.6,...,99.4,99.4,99.5,99.6,99.5,99.7,99.8,100.0,100.0,100.0
4,Algeria,DZA,"Access to electricity, rural (% of rural popul...",EG.ELC.ACCS.RU.ZS,97.4,97.4,97.40,97.4,97.4,97.4,...,98.1,98.3,98.6,98.9,98.7,99.1,99.3,99.3,100.0,100.0


In [32]:
#Filtering the data for selected indicators

selected_indicators = [
    "NY.GDP.MKTP.KD.ZG",
    "IT.NET.USER.ZS",
    "SE.SEC.ENRR"
]

df = df[df["Indicator Code"].isin(selected_indicators)]
df.shape

(138, 29)

In [33]:
# Reshaping the data from wide to long format

df_long = df.melt(
    id_vars=[
        "Country Name",
        "Country Code",
        "Indicator Name",
        "Indicator Code"
    ],
    var_name="Year",
    value_name="Value"
)
df_long.tail()

,Country Name,Country Code,Indicator Name,Indicator Code,Year,Value
3445,Zambia,ZMB,Individuals using the Internet (% of population),IT.NET.USER.ZS,2024,17.101000
3446,Zambia,ZMB,"School enrollment, secondary (% gross)",SE.SEC.ENRR,2024,56.788929
3447,Zimbabwe,ZWE,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,2024,1.676495
3448,Zimbabwe,ZWE,Individuals using the Internet (% of population),IT.NET.USER.ZS,2024,41.642799
3449,Zimbabwe,ZWE,"School enrollment, secondary (% gross)",SE.SEC.ENRR,2024,NaN


In [34]:
# Converting the "Year" column to integer and "Value" column to numeric

df_long["Year"] = df_long["Year"].astype(int)

df_long["Value"] = pd.to_numeric(
    df_long["Value"],
    errors="coerce"
)

In [35]:
# Checking for missing values in the "Value" column

missing_values = df_long["Value"].isnull().sum()
print(f"Missing values in 'Value' column: {missing_values}")

Missing values in 'Value' column: 595


In [36]:
missing_summary = (
    df_long.groupby("Indicator Code")["Value"]
    .apply(lambda x: x.isnull().mean()*100)
)

print(missing_summary)


Indicator Code
IT.NET.USER.ZS        4.260870
NY.GDP.MKTP.KD.ZG     3.913043
SE.SEC.ENRR          43.565217
Name: Value, dtype: float64


In [37]:
#pivot the data to have indicators as columns

master = df_long.pivot_table(
    index=[
        "Country Name",
        "Country Code",
        "Year"
    ],
    columns="Indicator Code",
    values="Value"
).reset_index()

#rename the columns for better readability
master.columns = [
    "Country",
    "Country_Code",
    "Year",
    "GDP_Growth",
    "Internet_Usage",
    "Secondary_Enrollment"
]

print(master.shape)
master.head()


(1134, 6)


,Country,Country_Code,Year,GDP_Growth,Internet_Usage,Secondary_Enrollment
0,Algeria,DZA,2000,0.491706,3.8,64.595421
1,Algeria,DZA,2001,0.646114,3.0,68.763298
2,Algeria,DZA,2002,1.591641,5.4,72.583069
3,Algeria,DZA,2003,2.195360,6.5,75.435867
4,Algeria,DZA,2004,4.634475,4.5,79.629860


In [38]:
#Remove rows with missing values in any of the selected indicators

master = master.dropna(
    how="all",
    subset=[
        "GDP_Growth",
        "Internet_Usage",
        "Secondary_Enrollment"
    ]
)
master.shape

(1134, 6)

In [39]:
#interpolate missing values for each country and indicator

master = master.sort_values(
    ["Country", "Year"]
)

for col in [
    "GDP_Growth",
    "Internet_Usage",
    "Secondary_Enrollment"
]:
    master[col] = (
        master.groupby("Country")[col]
        .transform(lambda x: x.interpolate())
    )
    

In [40]:
#Checking duplicate rows in the master dataframe

duplicates = master.duplicated(
    subset=["Country", "Year"]
).sum()

print(duplicates)

0


In [41]:
#Quality check: Check for any remaining missing values in the master dataframe

master.info()

master.describe()

master.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 1134 entries, 0 to 1133
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Country               1134 non-null   str    
 1   Country_Code          1134 non-null   str    
 2   Year                  1134 non-null   int64  
 3   GDP_Growth            1131 non-null   float64
 4   Internet_Usage        1120 non-null   float64
 5   Secondary_Enrollment  1054 non-null   float64
dtypes: float64(3), int64(1), str(2)
memory usage: 53.3 KB


Country                  0
Country_Code             0
Year                     0
GDP_Growth               3
Internet_Usage          14
Secondary_Enrollment    80
dtype: int64

In [42]:
master[master["Secondary_Enrollment"].isna()] \
    .sort_values(["Country", "Year"])

,Country,Country_Code,Year,GDP_Growth,Internet_Usage,Secondary_Enrollment
125,Burundi,BDI,2000,0.077248,-0.856864,NaN
175,Central African Republic,CAF,2000,0.053394,-2.489432,NaN
225,Comoros,COM,2000,0.271741,-1.007019,NaN
226,Comoros,COM,2001,0.443068,5.419600,NaN
250,Djibouti,DJI,2000,0.194501,NaN,NaN
...,...,...,...,...,...,...
1103,Zambia,ZMB,2019,14.471900,1.441306,NaN
1104,Zambia,ZMB,2020,14.645800,-2.785055,NaN
1105,Zambia,ZMB,2021,14.821900,6.234922,NaN
1106,Zambia,ZMB,2022,15.000000,5.211224,NaN


In [43]:
#Interpolate missing values for each country and indicator, forward fill and backward fill if necessary

for col in [
    "GDP_Growth",
    "Internet_Usage",
    "Secondary_Enrollment"
]:
    master[col] = (
        master.groupby("Country")[col]
        .transform(
            lambda x:
            x.interpolate()
             .ffill()
             .bfill()
        )
    )
master.isnull().sum()

Country                  0
Country_Code             0
Year                     0
GDP_Growth               0
Internet_Usage           0
Secondary_Enrollment    25
dtype: int64

In [44]:
master.to_csv(r"D:\GIT\data-analytics-portfolio\01-world-bank-analysis\data\cleaned_data\africa_data_cleaned.csv", index=False)